# Notebook used to test the models

### Packages Import


In [2]:
%load_ext autoreload
%autoreload 2

from ml_powerlifting_potential.data import load_dataset
from sklearn.model_selection import train_test_split
from ml_powerlifting_potential.config import FEATURES, TARGETS, SEED
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
import time

### Data Import & Train/Test Split

In [3]:
dataset = load_dataset()
dataset.describe()  # min, max, moyenne, quartiles de chaque colonne
dataset.isna().sum()  # doit afficher 0 partout

X = dataset[FEATURES]
y_squat = dataset[TARGETS[0]]
y_deadlift = dataset[TARGETS[1]]
X_train, X_test, y_squat_train, y_squat_test, y_deadlift_train, y_deadlift_test = (
    train_test_split(X, y_squat, y_deadlift, test_size=0.2, random_state=SEED)
)

bench_train = X_train["Best3BenchKg"]
bench_test = X_test["Best3BenchKg"]

### Error Evaluation Method

In [4]:
def evaluate(y_true, y_pred) -> dict[str, float]:
    """MAE in kg (average error) and R2 (1 = perfect, 0 = no better than the mean)."""
    return {"MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred)}

### Ratio Estimator Method

In [16]:
# Gym rule: median of the per-athlete ratios, computed on the TRAIN set only
ratio_squat = (y_squat_train / bench_train).median()
ratio_deadlift = (y_deadlift_train / bench_train).median()
print(f"squat/bench = {ratio_squat:.2f}, deadlift/bench = {ratio_deadlift:.2f}")

# Predictions on the TEST set
pred_squat_rule = bench_test * ratio_squat
pred_deadlift_rule = bench_test * ratio_deadlift


print("Squat rule:", evaluate(y_squat_test, pred_squat_rule))
print("Deadlift rule:", evaluate(y_deadlift_test, pred_deadlift_rule))

squat/bench = 1.57, deadlift/bench = 1.85
Squat rule: {'MAE': 21.590190954021267, 'R2': 0.7480737990839819}
Deadlift rule: {'MAE': 28.404329357495882, 'R2': 0.5838059452247737}


### Linear Regression Model

In [5]:
linear_model_squat = LinearRegression()
linear_model_squat.fit(X_train, y_squat_train)

linear_model_deadlift = LinearRegression()
linear_model_deadlift.fit(X_train, y_deadlift_train)

pred_squat_linear = linear_model_squat.predict(X_test)
pred_deadlift_linear = linear_model_deadlift.predict(X_test)

print("Squat linear:", evaluate(y_squat_test, pred_squat_linear))
print("Deadlift linear:", evaluate(y_deadlift_test, pred_deadlift_linear))

Squat linear: {'MAE': 17.624369244378496, 'R2': 0.8310389597243819}
Deadlift linear: {'MAE': 19.318098649301696, 'R2': 0.8073293069867008}


### Polynomial Regression Model

In [6]:
def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial features -> standardization -> linear regression."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("scale", StandardScaler()),
            ("linreg", LinearRegression()),
        ]
    )


targets = {
    "squat": (y_squat_train, y_squat_test),
    "deadlift": (y_deadlift_train, y_deadlift_test),
}

results = []
for degree in [2, 3]:
    for target_name, (y_train, y_test) in targets.items():
        model = make_poly_pipeline(degree)
        model.fit(X_train, y_train)
        for split_name, X_part, y_part in [
            ("train", X_train, y_train),
            ("test", X_test, y_test),
        ]:
            scores = evaluate(y_part, model.predict(X_part))
            results.append(
                {
                    "model": f"poly{degree}",
                    "target": target_name,
                    "split": split_name,
                    **scores,
                }
            )

results_df = pd.DataFrame(results)
results_df

,model,target,split,MAE,R2
0,poly2,squat,train,17.329804,0.835871
1,poly2,squat,test,17.343826,0.836093
2,poly2,deadlift,train,18.673604,0.819935
3,poly2,deadlift,test,18.720340,0.818673
4,poly3,squat,train,17.279228,0.836732
5,poly3,squat,test,17.315425,0.836684
6,poly3,deadlift,train,18.578508,0.821771
7,poly3,deadlift,test,18.640255,0.820338


### Support Vector Machine

In [7]:
def make_svr(
    C: float = 10, gamma: float = 0.2, epsilon: float = 0.1
) -> TransformedTargetRegressor:
    """RBF SVR with standardized inputs AND standardized target."""
    return TransformedTargetRegressor(
        regressor=Pipeline(
            [
                ("scale", StandardScaler()),
                ("svr", SVR(kernel="rbf", C=C, gamma=gamma, epsilon=epsilon)),
            ]
        ),
        transformer=StandardScaler(),  # target scaled for training, converted back to kg for predictions
    )

### Finding the best SVR params using GridSearch for the Squat Prediction

In [9]:
# values to try for each setting
param_grid = {
    "regressor__svr__C": [1, 10, 100],
    "regressor__svr__gamma": [0.05, 0.2, 0.5],
    "regressor__svr__epsilon": [0.05, 0.1],
}

# try every combination with 3-fold cross validation
grid_search_squat = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",  # negative because sklearn looks for the highest score
    n_jobs=-1,  # use all cpu cores
)

# tune on 8000 lifters only, the full train set would take too long
grid_search_squat.fit(X_train.iloc[:8_000], y_squat_train.iloc[:8_000])
best_SVR_C, best_SVR_epsilon, best_SVR_gamma = grid_search_squat.best_params_.values()
print("Best SVR parameters for squat:")
print(grid_search_squat.best_params_)
print("Best MAE (kg):")
print(-grid_search_squat.best_score_)  # best MAE in kg

Best SVR parameters for squat:
{'regressor__svr__C': 10, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.05}
Best MAE (kg):
17.25359349726295


In [10]:
# second pass, grid moved toward smaller gamma and bigger epsilon
param_grid = {
    "regressor__svr__C": [5, 10, 30],
    "regressor__svr__gamma": [0.01, 0.02, 0.05],
    "regressor__svr__epsilon": [0.1, 0.2],
}

grid_search_squat = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search_squat.fit(X_train.iloc[:8_000], y_squat_train.iloc[:8_000])

best_SVR_C, best_SVR_epsilon, best_SVR_gamma = grid_search_squat.best_params_.values()

print("Best SVR parameters for squat:")
print(grid_search_squat.best_params_)
print("Best MAE (kg):")
print(-grid_search_squat.best_score_)

Best SVR parameters for squat:
{'regressor__svr__C': 30, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.02}
Best MAE (kg):
17.22780060294379


### Finding the best SVR params using GridSearch for the Deadlift prediction

In [13]:
# small grid around the squat best params
param_grid = {
    "regressor__svr__C": [10, 30, 100],
    "regressor__svr__gamma": [0.02, 0.05],
    "regressor__svr__epsilon": [0.1],
}

grid_search_deadlift = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search_deadlift.fit(X_train.iloc[:8_000], y_deadlift_train.iloc[:8_000])

print("Best SVR parameters for deadlift:")
print(grid_search_deadlift.best_params_)
print("Best MAE (kg):")
print(-grid_search_deadlift.best_score_)

# have a look at the best params for squat and deadlift, they are very similar, so we can use the same SVR for both lifts

Best SVR parameters for deadlift:
{'regressor__svr__C': 30, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.02}
Best MAE (kg):
18.688749590269236


### Setting the best params for both Squat and Deadlift predictions

In [12]:
# best svr from the grid search, trained on 20000 lifters
best_svr_squat = make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon)
best_svr_squat.fit(X_train.iloc[:20_000], y_squat_train.iloc[:20_000])

best_svr_deadlift = make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon)
best_svr_deadlift.fit(X_train.iloc[:20_000], y_deadlift_train.iloc[:20_000])

# evaluate on the full test set
print("Squat SVR:", evaluate(y_squat_test, best_svr_squat.predict(X_test)))
print("Deadlift SVR:", evaluate(y_deadlift_test, best_svr_deadlift.predict(X_test)))

Squat SVR: {'MAE': 17.30984299163326, 'R2': 0.8363328725008987}
Deadlift SVR: {'MAE': 18.621181842392254, 'R2': 0.8201231225879759}


### Evaluation of different processing time 

In [14]:
targets = {
    "squat": (y_squat_train, y_squat_test),
    "deadlift": (y_deadlift_train, y_deadlift_test),
}

# name, how to build the model, number of lifters used to train it (None = all)
model_builders = [
    ("linear", lambda: LinearRegression(), None),
    ("poly2", lambda: make_poly_pipeline(2), None),
    ("poly3", lambda: make_poly_pipeline(3), None),
    ("svr", lambda: make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon), 20_000),
]

final_results = []
for target_name, (y_train, y_test) in targets.items():

    # gym rule: median ratio computed on the train set
    ratio = (y_train / X_train["Best3BenchKg"]).median()
    scores = evaluate(y_test, X_test["Best3BenchKg"] * ratio)
    final_results.append({"model": "gym rule", "target": target_name, "n_train": len(X_train), "fit_seconds": 0, **scores})

    for model_name, build_model, n_train in model_builders:
        n = n_train or len(X_train)
        model = build_model()

        start = time.perf_counter()
        model.fit(X_train.iloc[:n], y_train.iloc[:n])
        fit_seconds = time.perf_counter() - start

        scores = evaluate(y_test, model.predict(X_test))   # full test set
        final_results.append({"model": model_name, "target": target_name, "n_train": n, "fit_seconds": fit_seconds, **scores})

final_results_df = pd.DataFrame(final_results).round(3)
final_results_df

,model,target,n_train,fit_seconds,MAE,R2
0,gym rule,squat,182100,0.000,21.590,0.748
1,linear,squat,182100,0.024,17.624,0.831
2,poly2,squat,182100,0.229,17.344,0.836
3,poly3,squat,182100,0.713,17.315,0.837
4,svr,squat,20000,15.166,17.310,0.836
5,gym rule,deadlift,182100,0.000,28.404,0.584
6,linear,deadlift,182100,0.027,19.318,0.807
7,poly2,deadlift,182100,0.249,18.720,0.819
8,poly3,deadlift,182100,0.783,18.640,0.820
9,svr,deadlift,20000,13.376,18.621,0.820


### Application on personnal use

In [30]:
# my profile, same column order as FEATURES

# Personnal profile for prediction
BENCH_KG = 110
BODYWEIGHT_KG = 70
AGE = 20
SEX = 1
TESTED = 1

me = pd.DataFrame(
    [{"Best3BenchKg": BENCH_KG, "BodyweightKg": BODYWEIGHT_KG, "Age": AGE, "Sex": SEX, "Tested": TESTED}]
)

# train the best simple model on the full train set
poly_squat = make_poly_pipeline(2).fit(X_train, y_squat_train)
poly_deadlift = make_poly_pipeline(2).fit(X_train, y_deadlift_train)

print("Predicted squat:", poly_squat.predict(me[FEATURES])[0].round(1), "kg")
print("Predicted deadlift:", poly_deadlift.predict(me[FEATURES])[0].round(1), "kg")

# for comparison: the gym rule
print("Gym rule squat:", round(115 * ratio_squat, 1), "kg")
print("Gym rule deadlift:", round(115 * ratio_deadlift, 1), "kg")

Predicted squat: 166.3 kg
Predicted deadlift: 199.3 kg
Gym rule squat: 180.3 kg
Gym rule deadlift: 212.8 kg
